# Metodologia Design Science Research (DSR)

**Etapa de Pesquisa (Peffers et al., 2007):**
### 3. Design e Desenvolvimento (Design and Development)

**Objetivo Acadêmico:** Este notebook consolida o "oceano de dados" coletado nos notebooks 01a a 01e em um único artefato analítico (Base Mestra). A unificação de bases heterogêneas (estruturadas e não estruturadas) exige um rigoroso processo de limpeza e alinhamento temporal, garantindo a integridade dos dados necessária para o ciclo de indução de modelos de IA, conforme preconizado por Hevner et al. (2004).


# 02 - Consolidação e Limpeza Rigorosa
Este notebook unifica todas as fontes de dados e aplica filtros estritos de qualidade:
1. Apenas dias letivos conforme o calendário.
2. Apenas dias com cardápio válido (proteína identificada).
3. Apenas dias com pelo menos 1 reserva e 1 consumo registrado.

In [11]:
import pandas as pd
import numpy as np
import os

# Configurações
DATA_DIR = '../data/'
ARQUIVOS = {
    'calendario': 'calendario_consolidado.csv',
    'clima': 'clima_consolidado.csv',
    'cardapio': 'cardapio_consolidado.csv',
    'consumo': 'consumo_anonimo.csv',
    'reservas': 'reservas_anonimas.csv'
}
SAIDA_MESTRA = os.path.join(DATA_DIR, 'base_mestra_consolidada.csv')

In [12]:
# 1. Carregar Datasets
dfs = {}
for nome, arq in ARQUIVOS.items():
    path = os.path.join(DATA_DIR, arq)
    if os.path.exists(path):
        dfs[nome] = pd.read_csv(path)
        dfs[nome]['data'] = pd.to_datetime(dfs[nome]['data'])
        print(f"✅ {nome.capitalize()} carregado: {len(dfs[nome])} linhas.")

✅ Calendario carregado: 1096 linhas.
✅ Clima carregado: 1183 linhas.
✅ Cardapio carregado: 400 linhas.
✅ Consumo carregado: 59193 linhas.
✅ Reservas carregado: 58731 linhas.


In [13]:
# 2. Agregação de Reservas e Consumo
# Cruzamento Individual para fluxo
df_fluxo = pd.merge(
    dfs['reservas'][['data', 'id_aluno_anonimo']], 
    dfs['consumo'][['data', 'id_aluno_anonimo']], 
    on=['data', 'id_aluno_anonimo'], 
    how='outer', 
    indicator=True
)

metricas_diarias = df_fluxo.groupby('data')['_merge'].agg(
    reservou_e_comeu=lambda x: (x == 'both').sum(),
    reservou_e_nao_comeu=lambda x: (x == 'left_only').sum(),
    nao_reservou_e_comeu=lambda x: (x == 'right_only').sum()
).reset_index()

df_res_daily = dfs['reservas'].groupby('data').size().reset_index(name='total_reservas')
df_con_daily = dfs['consumo'].groupby('data').size().reset_index(name='total_servido')

df_consolidado_diario = pd.merge(df_res_daily, df_con_daily, on='data', how='outer')
df_consolidado_diario = pd.merge(df_consolidado_diario, metricas_diarias, on='data', how='outer').fillna(0)

In [14]:
# 3. Join Centralizado e Auditoria por Ano
# Unificar calendários letivos (base de tempo de interesse)
df_mestra = dfs['calendario'].copy()
df_mestra = pd.merge(df_mestra, dfs['clima'], on='data', how='left')
df_mestra = pd.merge(df_mestra, dfs['cardapio'], on='data', how='left')
df_mestra = pd.merge(df_mestra, df_consolidado_diario, on='data', how='left')

# Criar Relatório de Gaps para Dias Letivos (tem_refeicao == 1)
df_gaps = df_mestra[df_mestra['tem_refeicao'] == 1].copy()
df_gaps['ano'] = df_gaps['data'].dt.year

def get_tipo_dia(row):
    if row.get('eh_reuniao_impacto') == 1: return '📝 REUNIÃO'
    if row.get('eh_evento_especial') == 1: return '🎉 EVENTO'
    return '📖 LETIVO'

def check_status(row, col_critica):
    val = row.get(col_critica)
    if pd.isna(val) or val == 0 or val == 'SEM_CONTEUDO' or str(val).strip() == '':
        return '❌'
    return '✅'

relatorio = pd.DataFrame({
    'Data': df_gaps['data'].dt.strftime('%d/%m/%Y'),
    'Ano': df_gaps['ano'],
    'Tipo': df_gaps.apply(get_tipo_dia, axis=1),
    'Reserva': df_gaps.apply(lambda r: check_status(r, 'total_reservas'), axis=1),
    'Consumo': df_gaps.apply(lambda r: check_status(r, 'total_servido'), axis=1),
    'Cardápio': df_gaps.apply(lambda r: check_status(r, 'proteina_principal'), axis=1),
    'Clima': df_gaps.apply(lambda r: check_status(r, 'temp_media'), axis=1),
    'Evento/Obs': df_gaps['evento'].fillna('-')
})

# REGRA V5: Ignorar dias sem Reserva E sem Consumo (provável sem aula/refeição)
mask_presenca = (relatorio['Reserva'] == '✅') | (relatorio['Consumo'] == '✅')
relatorio_focado = relatorio[mask_presenca].copy()

print(f"--- RELATÓRIO DE AUDITORIA POR ANO (FOCO EM PRESENÇA) ---")
print(f"Filtro aplicado: Dias com PELO MENOS um registro de Reserva ou Consumo.")

for ano in sorted(relatorio['Ano'].unique()):
    df_ano = relatorio_focado[relatorio_focado['Ano'] == ano]
    df_ignore = relatorio[(relatorio['Ano'] == ano) & (~mask_presenca)]
    
    print(f"\n>>>> ANO {ano} <<<<")
    print(f"✅ Dias 100% OK: {len(df_ano[(df_ano['Reserva'] == '✅') & (df_ano['Consumo'] == '✅') & (df_ano['Cardápio'] == '✅')])}")
    print(f"⚠️ Dias com Pendências Parciais: {len(df_ano[df_ano.drop(['Data', 'Ano', 'Tipo', 'Evento/Obs'], axis=1).eq('❌').any(axis=1)])}")
    print(f"ℹ️ Dias Ignorados (Sem Reserva e Sem Consumo): {len(df_ignore)}")
    
    if len(df_ano) > 0:
        print(f"Amostra de Pendências em {ano}:")
        display(df_ano[df_ano.drop(['Data', 'Ano', 'Tipo', 'Evento/Obs'], axis=1).eq('❌').any(axis=1)].head(10))
    else:
        print(f"Nenhum dado de presença encontrado para {ano} no conjunto atual.")

--- RELATÓRIO DE AUDITORIA POR ANO (FOCO EM PRESENÇA) ---
Filtro aplicado: Dias com PELO MENOS um registro de Reserva ou Consumo.

>>>> ANO 2023 <<<<
✅ Dias 100% OK: 53
⚠️ Dias com Pendências Parciais: 140
ℹ️ Dias Ignorados (Sem Reserva e Sem Consumo): 57
Amostra de Pendências em 2023:


,Data,Ano,Tipo,Reserva,Consumo,Cardápio,Clima,Evento/Obs
36,06/02/2023,2023,📖 LETIVO,✅,✅,❌,✅,DIA LETIVO PADRAO
37,07/02/2023,2023,📖 LETIVO,✅,✅,❌,✅,DIA LETIVO PADRAO
38,08/02/2023,2023,📖 LETIVO,✅,✅,❌,✅,DIA LETIVO PADRAO
39,09/02/2023,2023,📖 LETIVO,✅,✅,❌,✅,DIA LETIVO PADRAO
40,10/02/2023,2023,📖 LETIVO,✅,✅,❌,✅,DIA LETIVO PADRAO
47,17/02/2023,2023,📖 LETIVO,✅,✅,❌,✅,DIA LETIVO PADRAO
52,22/02/2023,2023,📖 LETIVO,✅,❌,✅,✅,DIA LETIVO PADRAO
54,24/02/2023,2023,📖 LETIVO,✅,✅,❌,✅,DIA LETIVO PADRAO
59,01/03/2023,2023,📖 LETIVO,✅,✅,❌,✅,DIA LETIVO PADRAO
60,02/03/2023,2023,📖 LETIVO,✅,✅,❌,✅,DIA LETIVO PADRAO



>>>> ANO 2024 <<<<
✅ Dias 100% OK: 53
⚠️ Dias com Pendências Parciais: 100
ℹ️ Dias Ignorados (Sem Reserva e Sem Consumo): 77
Amostra de Pendências em 2024:


,Data,Ano,Tipo,Reserva,Consumo,Cardápio,Clima,Evento/Obs
410,15/02/2024,2024,📖 LETIVO,❌,✅,❌,✅,ENCONTRO DA CSP COM OS ESTUDANTES DO INTEGRADO...
411,16/02/2024,2024,📖 LETIVO,❌,✅,❌,✅,ENCONTRO DA CSP COM OS ESTUDANTES DO INTEGRADO...
414,19/02/2024,2024,📖 LETIVO,❌,✅,❌,✅,DIA LETIVO PADRAO
415,20/02/2024,2024,📖 LETIVO,❌,✅,❌,✅,DIA LETIVO PADRAO
416,21/02/2024,2024,📖 LETIVO,❌,✅,❌,✅,DIA LETIVO PADRAO
417,22/02/2024,2024,📖 LETIVO,❌,✅,❌,✅,DIA LETIVO PADRAO
418,23/02/2024,2024,📖 LETIVO,❌,✅,❌,✅,DIA LETIVO PADRAO
421,26/02/2024,2024,📖 LETIVO,❌,✅,❌,✅,DIA LETIVO PADRAO
422,27/02/2024,2024,📖 LETIVO,❌,✅,❌,✅,DIA LETIVO PADRAO
423,28/02/2024,2024,📖 LETIVO,❌,✅,❌,✅,DIA LETIVO PADRAO



>>>> ANO 2025 <<<<
✅ Dias 100% OK: 92
⚠️ Dias com Pendências Parciais: 87
ℹ️ Dias Ignorados (Sem Reserva e Sem Consumo): 42
Amostra de Pendências em 2025:


,Data,Ano,Tipo,Reserva,Consumo,Cardápio,Clima,Evento/Obs
780,19/02/2025,2025,📖 LETIVO,❌,✅,✅,✅,DIA LETIVO PADRAO
781,20/02/2025,2025,📖 LETIVO,❌,✅,✅,✅,DIA LETIVO PADRAO
782,21/02/2025,2025,📖 LETIVO,❌,✅,✅,✅,DIA LETIVO PADRAO
794,05/03/2025,2025,📖 LETIVO,✅,❌,✅,✅,DIA LETIVO PADRAO
802,13/03/2025,2025,📖 LETIVO,✅,❌,✅,✅,DIA LETIVO PADRAO
803,14/03/2025,2025,📖 LETIVO,✅,❌,✅,✅,DIA LETIVO PADRAO
838,18/04/2025,2025,📖 LETIVO,✅,❌,✅,✅,DIA LETIVO PADRAO
841,21/04/2025,2025,📖 LETIVO,✅,❌,✅,✅,DIA LETIVO PADRAO
865,15/05/2025,2025,📖 LETIVO,✅,❌,✅,✅,DIA LETIVO PADRAO
866,16/05/2025,2025,📖 LETIVO,✅,❌,✅,✅,DIA LETIVO PADRAO


In [15]:
# 4. Limpeza e Exportação Estrita (Apenas 100% OK)
# Filtrar apenas o que o modelo pode usar (dados completos)
mask_limpeza = (
    (df_mestra['tem_refeicao'] == 1) & 
    (df_mestra['total_reservas'] > 0) &
    (df_mestra['total_servido'] > 0) &
    (pd.notna(df_mestra['proteina_principal'])) &
    (df_mestra['proteina_principal'] != 'SEM_CONTEUDO') &
    (pd.notna(df_mestra['temp_media']))
)

df_final = df_mestra[mask_limpeza].copy()

# Tratar outros nulos remanescentes em colunas não críticas
df_final = df_final.fillna(method='ffill').fillna(0)

# Exportação
df_final.to_csv(SAIDA_MESTRA, index=False)

print(f"✨ Consolidação concluída.")
print(f"📊 Base Mestra final: {len(df_final)} registros aproveitados.")
if not df_final.empty:
    print(f"📅 Período coberto: {df_final['data'].min().strftime('%d/%m/%Y')} até {df_final['data'].max().strftime('%d/%m/%Y')}")

✨ Consolidação concluída.
📊 Base Mestra final: 198 registros aproveitados.
📅 Período coberto: 13/02/2023 até 22/08/2025


C:\Users\miche\AppData\Local\Temp\ipykernel_21164\1144742445.py:15: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df_final = df_final.fillna(method='ffill').fillna(0)
